# 01 — Data Exploration

This notebook covers:
- Loading and inspecting the raw dataset
- Data quality audit (missing values, duplicates, dtypes)
- Univariate distributions of key columns
- Target variable overview (class balance)

In [ ]:
import sys
import os

# Ensure project root is on the path
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    DATA_RAW_PATH,
    APPLICATION_DATA_FILE,
    KEY_NUMERIC_COLUMNS,
    KEY_CATEGORICAL_COLUMNS,
    FIGURE_SIZE,
)
from src.data_loader import load_application_data, validate_data_quality
from src.visualizations import plot_missing_values, plot_target_distribution, plot_numerical_distributions

%matplotlib inline
sns.set_theme(style='whitegrid')
print('Libraries loaded successfully.')

## 1. Load Data

In [ ]:
data_path = os.path.join(DATA_RAW_PATH, APPLICATION_DATA_FILE)

# Load the dataset (falls back to synthetic data for demo if file is absent)
try:
    df = load_application_data(data_path)
    print(f'Loaded {len(df):,} rows and {df.shape[1]} columns from {data_path}')
except FileNotFoundError:
    print('Real data not found. Creating synthetic sample for demonstration.')
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'TARGET': np.random.choice([0, 1], size=n, p=[0.92, 0.08]),
        'AMT_INCOME_TOTAL': np.random.lognormal(11, 0.5, n),
        'AMT_CREDIT': np.random.lognormal(12, 0.5, n),
        'AMT_ANNUITY': np.random.lognormal(9, 0.4, n),
        'AMT_GOODS_PRICE': np.random.lognormal(12, 0.5, n),
        'DAYS_BIRTH': -np.random.randint(7000, 25000, n),
        'DAYS_EMPLOYED': -np.random.randint(0, 10000, n),
        'CNT_CHILDREN': np.random.randint(0, 5, n),
        'CNT_FAM_MEMBERS': np.random.randint(1, 6, n),
        'CODE_GENDER': np.random.choice(['M', 'F'], n),
        'NAME_INCOME_TYPE': np.random.choice(
            ['Working', 'Commercial associate', 'Pensioner', 'State servant', 'Unemployed'],
            n, p=[0.5, 0.2, 0.2, 0.08, 0.02]
        ),
        'NAME_EDUCATION_TYPE': np.random.choice(
            ['Secondary / secondary special', 'Higher education', 'Incomplete higher', 'Lower secondary'],
            n, p=[0.7, 0.2, 0.07, 0.03]
        ),
        'OCCUPATION_TYPE': np.where(
            np.random.random(n) < 0.3, np.nan,
            np.random.choice(['Laborers', 'Core staff', 'Sales staff', 'Managers'], n)
        ),
    })
    print(f'Synthetic dataset created: {len(df):,} rows, {df.shape[1]} columns')

## 2. Basic Shape & Dtypes

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nDtype summary:')
print(df.dtypes.value_counts())
df.head()

## 3. Data Quality Audit

In [ ]:
quality_report = validate_data_quality(df)
print(f"Total rows: {quality_report['total_rows']:,}")
print(f"Total columns: {quality_report['total_columns']}")
print(f"Duplicate rows: {quality_report['duplicate_rows']}")
print(f"\nColumns exceeding 50% missing threshold:")
high_missing = quality_report['missing_by_column']
high_missing = high_missing[high_missing['missing_pct'] > 50]
print(f"{len(high_missing)} columns have >50% missing values")
high_missing.head(10)

In [ ]:
# Missing values visualisation
numeric_cols = [c for c in KEY_NUMERIC_COLUMNS if c in df.columns]
missing_subset = df[numeric_cols]
fig = plot_missing_values(missing_subset)
plt.tight_layout()
plt.show()

## 4. Target Variable Distribution

In [ ]:
default_rate = df['TARGET'].mean()
print(f'Default rate: {default_rate:.2%}')
print(f"Class counts:\n{df['TARGET'].value_counts()}")

fig = plot_target_distribution(df)
plt.tight_layout()
plt.show()

## 5. Univariate Distributions (Numeric Features)

In [ ]:
numeric_cols_present = [c for c in KEY_NUMERIC_COLUMNS if c in df.columns]
if numeric_cols_present:
    fig = plot_numerical_distributions(df, numeric_cols_present)
    plt.tight_layout()
    plt.show()
else:
    print('No key numeric columns found in this dataset.')

## 6. Categorical Columns Overview

In [ ]:
cat_cols_present = [c for c in KEY_CATEGORICAL_COLUMNS if c in df.columns]
for col in cat_cols_present:
    print(f'\n{col} — {df[col].nunique()} unique values')
    print(df[col].value_counts(normalize=True).head(5).map('{:.1%}'.format))

---
**Next:** Proceed to `02_detailed_analysis.ipynb` for bivariate analysis and statistical tests.